# Epsilon Calibration Review and Rerun
This notebook packages the calibration fixes, behavior-based tests, two targeted review passes, and a rerun of the runtime-aligned epsilon calibration.

It uses the fixed runtime replay path in `src/thermal_equilibrium_model.py` and the updated analysis helpers in `scripts/epsilon_sensitivity_analysis.py`.

## 1. Load Calibration Artifacts and Runtime Learner Inputs
Load the calibration helpers, inspect representative learner inputs, and capture the currently configured epsilon values before any further analysis.

In [1]:
from pathlib import Path
import json
import statistics
import subprocess
import sys
import time
from pprint import pprint

REPO = Path.cwd()
if not (REPO / "src").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists():
            REPO = candidate
            break

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.epsilon_sensitivity_analysis import (
    PARAMS,
    build_reference_prediction,
    compute_signal_delta,
    create_model,
    run_calibration,
    find_target_epsilon,
    check_linearity,
 )
from src.thermal_constants import PhysicsConstants

model = create_model()
baseline_inputs = {
    name: build_reference_prediction(name)["context"]
    for name in PARAMS
}
configured_epsilons = {
    name: details["current_eps"]
    for name, details in PARAMS.items()
}

print(f"Repository: {REPO}")
print("Configured epsilons:")
pprint(configured_epsilons)
print("\nSample context shapes:")
for name, context in baseline_inputs.items():
    print(
        name,
        {
            "pv_history_len": len(context.get("pv_power_history", [])),
            "pv_forecast_len": len(context.get("pv_forecast", [])),
            "outdoor_forecast_len": len(context.get("outdoor_forecast", [])),
            "climate_mode": context.get("climate_mode"),
            "delta_t": context.get("delta_t"),
            "thermal_power": context.get("thermal_power"),
        },
    )

Repository: c:\Users\ZOJHILK\OneDrive - Carl Zeiss AG\Dokumente\Heizung\PI4\ml_heating\ml_heating_underfloor
Configured epsilons:
{'heat_loss_coefficient': 0.008,
 'outlet_effectiveness': 0.1,
 'pv_heat_weight': 0.0005,
 'slab_time_constant_hours': 0.5,
 'solar_lag_minutes': 5.0,
 'thermal_time_constant': 0.2,
 'tv_heat_weight': 0.1}

Sample context shapes:
thermal_time_constant {'pv_history_len': 18, 'pv_forecast_len': 4, 'outdoor_forecast_len': 4, 'climate_mode': 'heating', 'delta_t': 2.5, 'thermal_power': 1.8}
heat_loss_coefficient {'pv_history_len': 18, 'pv_forecast_len': 4, 'outdoor_forecast_len': 4, 'climate_mode': 'heating', 'delta_t': 2.5, 'thermal_power': 1.8}
outlet_effectiveness {'pv_history_len': 18, 'pv_forecast_len': 4, 'outdoor_forecast_len': 4, 'climate_mode': 'heating', 'delta_t': 2.5, 'thermal_power': 1.8}
pv_heat_weight {'pv_history_len': 18, 'pv_forecast_len': 4, 'outdoor_forecast_len': 4, 'climate_mode': 'heating', 'delta_t': 2.5, 'thermal_power': 1.8}
tv_heat_weig

## 2. Apply Concrete Fix Set to the Calibration Script
Show the focused diff for the learner replay and the calibration helper so the notebook documents exactly what changed before rerunning calibration.

In [8]:
diff_cmd = [
    "git",
    "diff",
    "--",
    "src/thermal_equilibrium_model.py",
    "scripts/epsilon_sensitivity_analysis.py",
    "tests/unit/test_learning_stability.py",
]
diff_output = subprocess.run(
    diff_cmd, cwd=REPO, capture_output=True, text=True, check=False
).stdout
print(diff_output[:12000])

diff --git a/src/thermal_equilibrium_model.py b/src/thermal_equilibrium_model.py
index 8284948..678aadb 100644
--- a/src/thermal_equilibrium_model.py
+++ b/src/thermal_equilibrium_model.py
@@ -533,13 +533,28 @@ class ThermalEquilibriumModel:
         except (TypeError, ValueError):
             return fallback
 
-    def _resolve_delta_t_floor(self, observed_delta_t: float) -> float:
-        """Prefer the observed loop delta-T when present, else fall back to HP channel state."""
-        if observed_delta_t >= 1.0:
-            return observed_delta_t
-        return self._get_channel_parameter_value(
-            "heat_pump", "delta_t_floor", 2.0
-        )
+    def _resolve_delta_t_floor(self, observed_delta_t: float, climate_mode: str = "heating") -> float:
+        """Prefer the observed loop delta-T when present, else fall back to HP channel state.
+
+        In cooling mode delta_t is negative. The returned value is always
+        the *absolute* magnitude of the floor so caller

## 3. Add Behavior-Based Regression Tests
Run the touched-slice regression tests that validate runtime signal magnitude and the cooling-mode replay path instead of exact literal constants.

In [2]:
test_cmd = [
    sys.executable,
    "-m",
    "pytest",
    "tests/unit/test_learning_stability.py",
    "-q",
    "--tb=short",
]
test_result = subprocess.run(
    test_cmd, cwd=REPO, capture_output=True, text=True, check=False
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
assert test_result.returncode == 0, test_result.stdout + test_result.stderr

........................                                                 [100%]
24 passed in 1.64s



## 4. Run a Tooling/Docs-Aware Review Pass
Scan the developer-facing artifacts for issues that affect reproducibility, notebook and CLI usage, and project documentation hygiene.

In [3]:
changelog_text = (REPO / "CHANGELOG.md").read_text(encoding="utf-8")
unreleased = changelog_text.split("## [0.2.0]", 1)[0]
tooling_findings = []
if unreleased.count("### Changed") > 1:
    tooling_findings.append(
        {
            "severity": "medium",
            "area": "docs",
            "finding": "CHANGELOG Unreleased contains multiple Changed sections.",
        }
    )
for package in ["numpy", "dotenv"]:
    try:
        __import__(package if package != "dotenv" else "dotenv")
    except ModuleNotFoundError:
        tooling_findings.append(
            {
                "severity": "medium",
                "area": "environment",
                "finding": f"Notebook/kernel dependency missing: {package}",
            }
        )
print(json.dumps(tooling_findings, indent=2))

[
  {
    "severity": "medium",
    "area": "docs",
    "finding": "CHANGELOG Unreleased contains multiple Changed sections."
  }
]


## 5. Run a Runtime-Learner-Only Review Pass
Evaluate only the learner replay path and the resulting signals, excluding tooling and documentation concerns.

In [4]:
calibration = run_calibration()
runtime_findings = []
for row in calibration["results"]:
    if row["current_signal"] < 0.05:
        runtime_findings.append(
            {
                "severity": "medium",
                "parameter": row["parameter"],
                "finding": f"Current epsilon signal is low ({row['current_signal']:.6f}°C).",
            }
        )
    if not row["target_reachable"]:
        runtime_findings.append(
            {
                "severity": "high",
                "parameter": row["parameter"],
                "finding": "Target calibration window is not reachable with the tested epsilon search range.",
            }
        )
cooling_slab_signal = compute_signal_delta(
    model,
    "slab_time_constant_hours",
    PhysicsConstants.SLAB_TIME_CONSTANT_EPSILON,
    build_reference_prediction("slab_time_constant_hours", climate_mode="cooling"),
)
if cooling_slab_signal <= 0.01:
    runtime_findings.append(
        {
            "severity": "high",
            "parameter": "slab_time_constant_hours",
            "finding": "Cooling-mode slab replay still has near-zero sensitivity.",
        }
    )
print(json.dumps(runtime_findings, indent=2))
print("Cooling slab signal:", cooling_slab_signal)

[
  {
    "severity": "medium",
    "parameter": "solar_lag_minutes",
    "finding": "Current epsilon signal is low (0.002126\u00b0C)."
  },
  {
    "severity": "high",
    "parameter": "solar_lag_minutes",
    "finding": "Target calibration window is not reachable with the tested epsilon search range."
  },
  {
    "severity": "medium",
    "parameter": "slab_time_constant_hours",
    "finding": "Current epsilon signal is low (0.035688\u00b0C)."
  }
]
Cooling slab signal: 0.011896130116358705


## 6. Execute Calibration End-to-End Again
Rerun the fixed calibration flow end to end and capture timing, warnings, and the returned structured results.

In [5]:
start = time.perf_counter()
calibration_rerun = run_calibration()
elapsed = time.perf_counter() - start
print(f"Calibration rerun completed in {elapsed:.3f}s")
for row in calibration_rerun["results"]:
    print(
        row["parameter"],
        {
            "current_epsilon": round(row["current_epsilon"], 6),
            "current_signal": round(row["current_signal"], 6),
            "recommended_epsilon": round(row["recommended_epsilon"], 6),
            "recommended_signal": round(row["recommended_signal"], 6),
            "target_reachable": row["target_reachable"],
        },
    )

Calibration rerun completed in 0.309s
thermal_time_constant {'current_epsilon': 0.2, 'current_signal': np.float64(0.190328), 'recommended_epsilon': np.float64(0.192593), 'recommended_signal': np.float64(0.183271), 'target_reachable': True}
heat_loss_coefficient {'current_epsilon': 0.008, 'current_signal': np.float64(0.185441), 'recommended_epsilon': np.float64(0.008887), 'recommended_signal': np.float64(0.205995), 'target_reachable': True}
outlet_effectiveness {'current_epsilon': 0.1, 'current_signal': np.float64(0.181179), 'recommended_epsilon': np.float64(0.110613), 'recommended_signal': np.float64(0.200797), 'target_reachable': True}
pv_heat_weight {'current_epsilon': 0.0005, 'current_signal': np.float64(0.249463), 'recommended_epsilon': np.float64(0.000391), 'recommended_signal': np.float64(0.195256), 'target_reachable': True}
tv_heat_weight {'current_epsilon': 0.1, 'current_signal': np.float64(0.099263), 'recommended_epsilon': np.float64(0.175), 'recommended_signal': np.float64(0.

## 7. Collect Metrics, Diffs, and Test Results
Summarize the calibration metrics, diff footprint, and the touched-slice regression outcomes in a compact machine-readable form.

In [6]:
summary = {
    "signals": {
        row["parameter"]: {
            "current": round(row["current_signal"], 6),
            "recommended": round(row["recommended_signal"], 6),
            "target_reachable": row["target_reachable"],
        }
        for row in calibration_rerun["results"]
    },
    "tooling_findings": tooling_findings,
    "runtime_findings": runtime_findings,
    "test_returncode": test_result.returncode,
    "diff_files": [
        "src/thermal_equilibrium_model.py",
        "scripts/epsilon_sensitivity_analysis.py",
        "tests/unit/test_learning_stability.py",
    ],
}
print(json.dumps(summary, indent=2))

{
  "signals": {
    "thermal_time_constant": {
      "current": 0.190328,
      "recommended": 0.183271,
      "target_reachable": true
    },
    "heat_loss_coefficient": {
      "current": 0.185441,
      "recommended": 0.205995,
      "target_reachable": true
    },
    "outlet_effectiveness": {
      "current": 0.181179,
      "recommended": 0.200797,
      "target_reachable": true
    },
    "pv_heat_weight": {
      "current": 0.249463,
      "recommended": 0.195256,
      "target_reachable": true
    },
    "tv_heat_weight": {
      "current": 0.099263,
      "recommended": 0.17371,
      "target_reachable": true
    },
    "solar_lag_minutes": {
      "current": 0.002126,
      "recommended": 0.009729,
      "target_reachable": false
    },
    "slab_time_constant_hours": {
      "current": 0.035688,
      "recommended": 0.121474,
      "target_reachable": true
    }
  },
  "tooling_findings": [
    {
      "severity": "medium",
      "area": "docs",
      "finding": "CHANGELO

## 8. Check Result Stability and Failure Cases
Repeat the calibration to confirm determinism and flag unstable metrics, unreachable targets, or mismatches between expected and observed learner behavior.

In [7]:
runs = [run_calibration() for _ in range(3)]
stability = {}
for parameter in PARAMS:
    signals = [
        run["results"][list(PARAMS).index(parameter)]["current_signal"]
        for run in runs
    ]
    recommendations = [
        run["results"][list(PARAMS).index(parameter)]["recommended_epsilon"]
        for run in runs
    ]
    stability[parameter] = {
        "signal_min": min(signals),
        "signal_max": max(signals),
        "signal_span": max(signals) - min(signals),
        "recommended_min": min(recommendations),
        "recommended_max": max(recommendations),
        "recommended_span": max(recommendations) - min(recommendations),
    }
print(json.dumps(stability, indent=2))
unstable = {
    name: stats
    for name, stats in stability.items()
    if stats["signal_span"] > 1e-9 or stats["recommended_span"] > 1e-9
}
print("Unstable parameters:", unstable)

{
  "thermal_time_constant": {
    "signal_min": 0.1903279344346025,
    "signal_max": 0.1903279344346025,
    "signal_span": 0.0,
    "recommended_min": 0.192593144600748,
    "recommended_max": 0.192593144600748,
    "recommended_span": 0.0
  },
  "heat_loss_coefficient": {
    "signal_min": 0.18544133338359714,
    "signal_max": 0.18544133338359714,
    "signal_span": 0.0,
    "recommended_min": 0.008886593957759812,
    "recommended_max": 0.008886593957759812,
    "recommended_span": 0.0
  },
  "outlet_effectiveness": {
    "signal_min": 0.18117908273778838,
    "signal_max": 0.18117908273778838,
    "signal_span": 0.0,
    "recommended_min": 0.11061254933884457,
    "recommended_max": 0.11061254933884457,
    "recommended_span": 0.0
  },
  "pv_heat_weight": {
    "signal_min": 0.24946265187567107,
    "signal_max": 0.24946265187567107,
    "signal_span": 0.0,
    "recommended_min": 0.00039110958020714594,
    "recommended_max": 0.00039110958020714594,
    "recommended_span": 0.0
 